# Dev/prod parity

**Objective:** Use the same code, dependency versions, and backing-service types in development and production.

## Simple version

Connection addresses may change between environments; the database and cache technologies should not.

In [ ]:
# Addresses may differ, but both environments should use the same service types.
development = {"database": "postgresql", "cache": "redis"}
production = {"database": "postgresql", "cache": "redis"}

print(development == production)

## Polished version

A parity check detects accidental environment drift while allowing deployment-specific addresses and replica counts.

In [ ]:
from dataclasses import dataclass


# Capture both the properties that must match and the addresses allowed to differ.
@dataclass(frozen=True)
class Environment:
    name: str
    python: str
    lock_digest: str
    database_kind: str
    database_url: str
    cache_kind: str
    cache_url: str


def verify_parity(first: Environment, second: Environment) -> None:
    # URLs are intentionally excluded because each deployment has its own address.
    comparable = (
        "python",
        "lock_digest",
        "database_kind",
        "cache_kind",
    )
    drift = [
        field
        for field in comparable
        if getattr(first, field) != getattr(second, field)
    ]
    if drift:
        raise ValueError(f"Environment drift: {', '.join(drift)}")


development = Environment(
    name="development",
    python="3.12",
    lock_digest="sha256:abc",
    database_kind="postgresql",
    database_url="postgresql://localhost/app",
    cache_kind="redis",
    cache_url="redis://localhost/0",
)
production = Environment(
    name="production",
    python="3.12",
    lock_digest="sha256:abc",
    database_kind="postgresql",
    database_url="postgresql://db.internal/app",
    cache_kind="redis",
    cache_url="rediss://redis.internal/0",
)

# Raise immediately if an important runtime property has drifted.
verify_parity(development, production)
print("Parity verified")

## Applied in this repository

Lockfiles pin dependencies, Docker Compose provides PostgreSQL or Redis locally, and environment variables change connection addresses without swapping technologies or code paths.